<a href="https://colab.research.google.com/github/nisaa783/scikit-learn-cookbook_Anisa-Khasanah/blob/main/Chapter_2_Pre_Model_Workflow_and_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Pendahuluan dan Pentingnya Pra-pemrosesan Data**
Dalam dunia nyata, data mentah (raw data) hampir tidak pernah siap untuk langsung dimasukkan ke dalam algoritma machine learning. Data sering kali mengandung nilai yang hilang (missing values), memiliki rentang skala nilai yang tidak seragam antar fitur (misalnya kolom umur berkisar 0-100, sementara kolom gaji berkisar juta-an), atau berupa data teks dan kategori (categorical variables) yang tidak dapat diproses secara matematis oleh komputer.Pra-pemrosesan data (data preprocessing) adalah tahapan krusial untuk membersihkan, mentransformasikan, dan menyiapkan data agar kualitasnya optimal, yang pada akhirnya akan meningkatkan performa dan stabilitas model machine learning.

## **2. Penanganan Nilai Kosong (Missing Values)**
Ketika dataset memiliki sel atau baris yang kosong (NaN), sebagian besar algoritma machine learning tidak dapat memprosesnya dan akan menghasilkan galat (error). Pendekatan paling umum selain menghapus baris yang kosong adalah imputasi, yaitu mengisi nilai yang hilang menggunakan estimasi statistik dari data yang ada di kolom tersebut, seperti:Mean (Rata-rata): Mengganti nilai kosong dengan rata-rata seluruh nilai pada kolom tersebut (cocok untuk data numerik tanpa outlier ekstrem).Median: Mengganti dengan nilai tengah (lebih tahan terhadap outlier).Most Frequent (Modus): Mengganti dengan nilai yang paling sering muncul (biasanya digunakan untuk data kategorikal).

## **3. Penskalaan Fitur (Feature Scaling)**
Algoritma machine learning yang berbasis jarak (seperti KNN, SVM) atau optimasi berbasis gradien (gradient descent seperti Regresi Linear dan Neural Network) sangat sensitif terhadap skala fitur. Jika satu fitur memiliki rentang nilai yang jauh lebih besar daripada fitur lainnya, algoritma akan menganggap fitur tersebut lebih penting secara keliru.Konsep & Rumus Matematika:StandardScaler (Z-score Normalization): Menskalakan data agar memiliki rata-rata ($\mu$) sama dengan 0 dan standar deviasi ($\sigma$) sama dengan 1.$$z = \frac{x - \mu}{\sigma}$$MinMaxScaler (Normalization): Menskalakan seluruh nilai data ke dalam rentang tertentu, biasanya antara 0 dan 1.$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

## **4. Encoding Variabel Kategorikal**
Sebagian besar algoritma matematika dalam machine learning memerlukan input berupa angka, bukan teks atau kategori (misalnya warna: "Merah", "Hijau", "Biru"). Kita memerlukan teknik encoding untuk mengubah data kategori ini menjadi representasi numerik:One-Hot Encoding: Mengubah setiap kategori unik menjadi kolom biner baru (kolom terpisah bernilai 0 atau 1).Ordinal Encoding: Mengubah kategori menjadi angka berurutan (misalnya Rendah = 1, Sedang = 2, Tinggi = 3), biasanya digunakan jika ada tingkatan yang jelas.

## **5. Contoh Program Penerapan 1: Imputasi Data Kosong dan Feature Scaling Menggunakan StandardScaler**
Contoh program pertama ini memperlihatkan bagaimana menangani missing values pada data numerik menggunakan SimpleImputer dan menyeragamkan skala fitur menggunakan StandardScaler.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Membuat dataset tiruan (simulasi data mentah dengan missing values)
data = {
    'Usia': [25, 30, np.nan, 45, 22, 35],
    'Pendapatan': [50000, np.nan, 80000, 120000, 45000, 95000]
}
df = pd.DataFrame(data)
print("Data Asli dengan Missing Values:")
print(df)

# 2. Imputasi Missing Values menggunakan nilai rata-rata (mean)
imputer = SimpleImputer(strategy='mean')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
print("\nData Setelah Imputasi (Mean):")
print(df_imputed)

# 3. Feature Scaling menggunakan StandardScaler (Z-score Normalization)
# Rumus: z = (x - mean) / std
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_imputed), columns=df.columns)

print("\nData Setelah Feature Scaling (StandardScaler):")
print(df_scaled)
print(f"\nRata-rata tiap kolom setelah scaling (mendekati 0):\n{df_scaled.mean().round(4)}")
print(f"Standar deviasi tiap kolom setelah scaling (mendekati 1):\n{df_scaled.std().round(4)}")

Data Asli dengan Missing Values:
   Usia  Pendapatan
0  25.0     50000.0
1  30.0         NaN
2   NaN     80000.0
3  45.0    120000.0
4  22.0     45000.0
5  35.0     95000.0

Data Setelah Imputasi (Mean):
   Usia  Pendapatan
0  25.0     50000.0
1  30.0     78000.0
2  31.4     80000.0
3  45.0    120000.0
4  22.0     45000.0
5  35.0     95000.0

Data Setelah Feature Scaling (StandardScaler):
           Usia  Pendapatan
0 -8.640238e-01   -1.094051
1 -1.890052e-01    0.000000
2 -4.796295e-16    0.078146
3  1.836050e+00    1.641076
4 -1.269035e+00   -1.289417
5  4.860134e-01    0.664245

Rata-rata tiap kolom setelah scaling (mendekati 0):
Usia         -0.0
Pendapatan    0.0
dtype: float64
Standar deviasi tiap kolom setelah scaling (mendekati 1):
Usia          1.0954
Pendapatan    1.0954
dtype: float64


## **6. Contoh Program Penerapan 2: Menggabungkan Preprocessing Numerik dan Kategorikal dengan ColumnTransformer**
Di dunia nyata, dataset biasanya berisi campuran antara kolom numerik dan kategorikal. Kita dapat menggunakan ColumnTransformer untuk menerapkan aturan pemrosesan yang berbeda secara spesifik pada masing-masing tipe kolom secara bersamaan.

In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Membuat dataset campuran (numerik dan kategorikal) dengan missing values
data = {
    'Umur': [22, 38, np.nan, 35, 28],
    'Kota': ['Jakarta', 'Bandung', 'Surabaya', np.nan, 'Jakarta'],
    'Gaji': [4000, 7200, 5000, 8500, np.nan]
}
df = pd.DataFrame(data)
print("Dataset Campuran Asli:")
print(df)

# 2. Menentukan kolom numerik dan kategorikal
numeric_features = ['Umur', 'Gaji']
categorical_features = ['Kota']

# 3. Membuat pipeline preprocessing untuk masing-masing tipe data
# Untuk numerik: isi kosong dengan median, lalu scaling
numeric_transformer = StandardScaler() # (Bisa digabung imputer jika diperlukan)

# Untuk kategorikal: isi kosong dengan string konstan, lalu One-Hot Encode
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 4. Menggabungkan transformer menggunakan ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Kolom lain yang tidak disebut akan dibiarkan
)

# Catatan: Kita tangani missing value numerik dulu dengan imputer sederhana pada kolom numerik
df_num_imputed = df.copy()
df_num_imputed['Umur'] = SimpleImputer(strategy='median').fit_transform(df[['Umur']])
df_num_imputed['Gaji'] = SimpleImputer(strategy='median').fit_transform(df[['Gaji']])
df_num_imputed['Kota'] = df['Kota'].fillna('Unknown') # Mengisi missing kategori

# Menjalankan ColumnTransformer
X_processed = preprocessor.fit_transform(df_num_imputed)

print("\nHasil Transformasi Data (Numerik diskalakan + One-Hot Encoding Kota):")
print(X_processed)

Dataset Campuran Asli:
   Umur      Kota    Gaji
0  22.0   Jakarta  4000.0
1  38.0   Bandung  7200.0
2   NaN  Surabaya  5000.0
3  35.0       NaN  8500.0
4  28.0   Jakarta     NaN

Hasil Transformasi Data (Numerik diskalakan + One-Hot Encoding Kota):
[[-1.5974569  -1.36218649  0.          1.          0.          0.        ]
 [ 1.27437573  0.65586757  1.          0.          0.          0.        ]
 [ 0.10769372 -0.73154459  0.          0.          1.          0.        ]
 [ 0.73590711  1.47570203  0.          0.          0.          1.        ]
 [-0.52051966 -0.03783851  0.          1.          0.          0.        ]]
